# 02 — Preprocessing Pipeline

Feature engineering, normalization, sliding-window construction, and options data preparation.

**Critical rules enforced here:**
- No lookahead: `.shift(1)` before all rolling windows
- Scaler fit on 2010-2019 only
- Strict temporal train/test split

**Outputs:** `data/processed/` — master_df.csv, scaler.pkl, train_sequences.npy, opts_train.pkl, opts_test.pkl

In [ ]:
import sys, os
import numpy as np
import pandas as pd
sys.path.insert(0, os.path.abspath('..'))

from src.data.download import load_cboe_files
from src.data.preprocess import (
    build_master_dataframe, clean_and_split, fit_scaler,
    scale_data, build_sequences, preprocess_options, split_options,
    FEATURE_COLS, SEQ_LEN
)

RAW_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

## Step 1: Load raw data

In [ ]:
spy = pd.read_csv(os.path.join(RAW_DIR, 'spy.csv'), index_col=0, parse_dates=True)
vix = pd.read_csv(os.path.join(RAW_DIR, 'vix.csv'), index_col=0, parse_dates=True)

vvix_path = os.path.join(RAW_DIR, 'vvix.csv')
vvix = pd.read_csv(vvix_path, index_col=0, parse_dates=True) if os.path.exists(vvix_path) else None

hyg = pd.read_csv(os.path.join(RAW_DIR, 'hyg.csv'), index_col=0, parse_dates=True)
lqd = pd.read_csv(os.path.join(RAW_DIR, 'lqd.csv'), index_col=0, parse_dates=True)
dxy = pd.read_csv(os.path.join(RAW_DIR, 'dxy.csv'), index_col=0, parse_dates=True)

treasury_path = os.path.join(RAW_DIR, 'treasury.csv')
treasury = pd.read_csv(treasury_path, index_col=0, parse_dates=True) if os.path.exists(treasury_path) else None

cboe = load_cboe_files(RAW_DIR)

print(f'SPY: {spy.shape}')
print(f'VIX: {vix.shape}')
print(f'VVIX: {vvix.shape if vvix is not None else "NOT FOUND"}')
print(f'Treasury: {treasury.shape if treasury is not None else "NOT FOUND"}')

## Step 2: Build master DataFrame with 20 features

In [ ]:
master = build_master_dataframe(spy, vix, vvix, hyg, lqd, dxy, treasury, cboe)
print(f'Master shape: {master.shape}')
print(f'Date range: {master.index[0].date()} to {master.index[-1].date()}')
print(f'\nNaN counts before dropping:')
print(master[FEATURE_COLS].isna().sum())
master.head()

## Step 3-4: Drop NaN rows and split

In [ ]:
train_df, test_df = clean_and_split(master)
print(f'Training period: {train_df.index[0].date()} to {train_df.index[-1].date()} ({len(train_df)} days)')
print(f'Testing period:  {test_df.index[0].date()} to {test_df.index[-1].date()} ({len(test_df)} days)')
print(f'\nTotal dropped NaN rows: {len(master) - len(train_df) - len(test_df)}')

## Step 5: Fit scaler on training data only

In [ ]:
scaler = fit_scaler(train_df, os.path.join(PROCESSED_DIR, 'scaler.pkl'))

train_scaled = scale_data(train_df, scaler)
test_scaled = scale_data(test_df, scaler)

print(f'Train scaled shape: {train_scaled.shape}')
print(f'Test scaled shape:  {test_scaled.shape}')
print(f'\nTrain mean (should be ~0): {train_scaled.mean(axis=0)[:5].round(4)}')
print(f'Train std  (should be ~1): {train_scaled.std(axis=0)[:5].round(4)}')

## Step 6: Build sliding-window sequences for GAN

In [ ]:
train_seqs = build_sequences(train_scaled, seq_len=SEQ_LEN)
print(f'Training sequences shape: {train_seqs.shape}')
print(f'  (expected: (~{len(train_scaled) - SEQ_LEN}, {SEQ_LEN}, 20))')

np.save(os.path.join(PROCESSED_DIR, 'train_sequences.npy'), train_seqs)
np.save(os.path.join(PROCESSED_DIR, 'train_scaled.npy'), train_scaled)
np.save(os.path.join(PROCESSED_DIR, 'test_scaled.npy'), test_scaled)
print('Saved to data/processed/')

## Step 7: Preprocess options data

In [ ]:
# Save the master DF (needed for options preprocessing and evaluation)
master.to_csv(os.path.join(PROCESSED_DIR, 'master_df.csv'))

# Save date indices for later
train_df.index.to_frame().to_csv(os.path.join(PROCESSED_DIR, 'train_dates.csv'))
test_df.index.to_frame().to_csv(os.path.join(PROCESSED_DIR, 'test_dates.csv'))

In [ ]:
# Load and preprocess options
opts_file = None
for fname in ['spy_options.csv', 'spy_options_data.csv', 'sp500_options.csv', 'options.csv']:
    path = os.path.join(RAW_DIR, fname)
    if os.path.exists(path):
        opts_file = path
        break

if opts_file:
    print(f'Loading options from {opts_file}...')
    opts_raw = pd.read_csv(opts_file, parse_dates=['date', 'expiration'])
    print(f'Raw options: {len(opts_raw)} rows')
    
    opts = preprocess_options(opts_raw, master, scaler, seq_len=SEQ_LEN)
    print(f'After filtering: {len(opts)} contracts')
    
    opts_train, opts_test = split_options(opts)
    print(f'Train options: {len(opts_train)} (pre-2020)')
    print(f'Test options:  {len(opts_test)} (2020-2023)')
    
    opts_train.to_pickle(os.path.join(PROCESSED_DIR, 'opts_train.pkl'))
    opts_test.to_pickle(os.path.join(PROCESSED_DIR, 'opts_test.pkl'))
    print('Saved to data/processed/')
else:
    print('OPTIONS FILE NOT FOUND. Place spy_options.csv in data/raw/ and re-run.')

## Verification

In [ ]:
import glob
files = sorted(glob.glob(os.path.join(PROCESSED_DIR, '*')))
print('Files in data/processed/:')
for f in files:
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {os.path.basename(f):30s}  {size_mb:>8.2f} MB')